### Adaptive RAG

In [19]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["LANGCHAIN_API_KEY"]=os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"]=os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_TRACING_V2"]="true"

In [20]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

In [21]:
urls=[
    "https://python.langchain.com/docs/tutorials/"
    ]

docs = [WebBaseLoader(url).load() for url in urls]
docs_list = [item for sublist in docs for item in sublist]
docs_list

[Document(metadata={'source': 'https://python.langchain.com/docs/tutorials/', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.', 'language': 'en'}, page_content='LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what\'s next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep AgentsLangChainLangGraphOpenWikiIntegrationsLearnReferenceContributePythonOverviewGet startedInstallQuickstartChangelogPhilosophyCore componentsAgentsModelsMessag

In [22]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
text_splitter

In [23]:
doc_split = text_splitter.split_documents(docs_list)
doc_split

[Document(metadata={'source': 'https://python.langchain.com/docs/tutorials/', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain provides create_agent: a minimal, highly configurable agent harness. Compose exactly the agent your use case needs from model, tools, prompt, and middleware.', 'language': 'en'}, page_content="LangChain overview - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentInterrupt is coming to NYC and London this fall. Join the builders, engineers, and teams shaping what's next for agents. Get your tickets →Docs by LangChain home pageBuildSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangChain overviewOverviewDeep AgentsManaged Deep"),
 Document(metadata={'source': 'https://python.langchain.com/docs/tutorials/', 'title': 'LangChain overview - Docs by LangChain', 'description': 'LangChain provi

In [24]:
vector_store = FAISS.from_documents(documents=doc_split, embedding=HuggingFaceEmbeddings())
retriever = vector_store.as_retriever()
retriever

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6015.53it/s]


VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000026F131C3BC0>, search_kwargs={})

In [25]:
### Router
from typing import Literal
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

from pydantic import BaseModel, Field

class RouteQuery(BaseModel):
     """Route a user query to the most relevant datasource."""

     datasource: Literal["vectorstore", "websearch"] = Field(description="Given a user question choose to route it to web search or a vectorstore")

# LLM with function call

llm = ChatGroq(model="openai/gpt-oss-120b")
structured_llm_output = llm.with_structured_output(RouteQuery)

# Prompt
system = """ you are an expert at routing a user question to a vector store or web search. The vector store contains documents related to 
agents, prompt engineering and adversarial attacks. use the vectorstore for questions on these topics, otherwise use web search"""

route_prompt = ChatPromptTemplate.from_messages([("system", system), ("human", "{question}")])

question_router = route_prompt | structured_llm_output

print(question_router.invoke({"question": "who won the cricket world cup 2023"}))


datasource='websearch'


In [26]:
print(question_router.invoke({"question":"what is an agent memory"}))

datasource='vectorstore'


In [27]:
### Retrieval Grader

# grade documents data model
class GradeDocuments(BaseModel):
    binary_score:str = Field(description="Documents are relevant to the question, 'yes' or 'no'")

llm_with_structured_output = llm.with_structured_output(GradeDocuments)

system = """ you are a grader assessing relevance of a retrieved document to a user question. if the document contains 
keyword or semantic meaning related to user question, grade it as relevant. it does not need to be stringent test. 
the goal is to filter out errouneous retrievals. give a binary score 'yes' or 'no' to indicate whether the document is relavant to the question. """

grade_prompt = ChatPromptTemplate.from_messages([("system", system), ("human", "Retrieved document: {document}, user question: {question}")])

retrieval_grade = grade_prompt | llm_with_structured_output

question = "what is agent memory"

docs = retriever.invoke(question)
retrieved_document = docs[0]
retrieval_grade.invoke({"question":question, "document":retrieved_document.page_content})


GradeDocuments(binary_score='yes')

In [28]:
### Generate
from langchain_core.output_parsers import StrOutputParser

from langsmith import Client

client = Client()

prompt = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

output_parser = StrOutputParser()

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = prompt | llm | output_parser

generation = rag_chain.invoke({"context": docs, "question": question})
generation


'Agent memory is the component that lets an agent retain information across turns, enabling it to recall past interactions or external knowledge when generating responses. It can be short‑term (e.g., conversation history) or long‑term (e.g., stored facts retrieved from a database). This memory is integrated into the agent’s loop so the model can use the accumulated context for more coherent behavior.'

In [29]:
# Hallucination Grader

class GradeHallucinations(BaseModel):
    """ Binary score for hallucinations present in generation answer """

    binary_score: str = Field(description="Answer is grounded in the facts, 'yes' or 'no'")

llm_with_structured_output= llm.with_structured_output(GradeHallucinations)

system = """ you are grader assessing whether an llm generation is grounded in / supported by a set of retrieved facts.
give a binary score 'yes' or 'no'. 'yes' means that the answer is grounded / supported by set of facts """

hallucination_prompt = ChatPromptTemplate.from_messages([("system", system), ("human", "set of facts {document}, llm generation {generation}")])

hallucination_grader = hallucination_prompt | llm_with_structured_output

hallucination_grader.invoke({"document": docs, "generation":generation})

GradeHallucinations(binary_score='no')

In [30]:
### Answer Grader

class AnswetGrader(BaseModel):
    """Binary score to assess answer addresses question."""

    binary_score:str = Field(description="Answer addresses the question, 'yes' or 'no'")

system = """ you are a grade assigner whether the genearted answer address the question,
give a binary score 'yes" or 'no', 'yes' means that answer resolves the question """

llm_with_structured_output = llm.with_structured_output(AnswetGrader)

answer_prompt = ChatPromptTemplate.from_messages([("system", system), ("human", "user question: {question} and generation {generation}")])

answer_grader = answer_prompt | llm_with_structured_output

answer_grader.invoke({"question": question, "generation":generation})

AnswetGrader(binary_score='yes')

In [31]:
### Question Re-writer

system = """ you are an question rewriter that converts an input question to a better version that is optimized for vector store retrieval,
look at the input and try to reason about the underlying semantic intent/ meaning """

re_write_prompt = ChatPromptTemplate.from_messages([
    ("system", system), ("human", "here is the initiated question {question} formulate an improved question")
])

question_rewriter_chain = re_write_prompt | llm | StrOutputParser()

question_rewriter_chain.invoke({"question", question})

'**Improved question:**  \nWhat does “agent memory” refer to in artificial intelligence, and how is it used by AI agents?'

In [32]:
### Search

from langchain_community.tools.tavily_search import TavilySearchResults

web_search_tool = TavilySearchResults(max_results=3)

In [33]:
from typing import List
from typing_extensions import TypedDict

class GraphState(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        question: question
        generation: LLM generation
        documents: list of documents
        loop_count: number of transform_query/generate retries so far
    """

    question:str
    generation:str
    documents: List[str]
    loop_count: int


In [34]:
from langchain_core.documents import Document
from pprint import pprint

MAX_RETRIES = 3

def retrieve(state):
    """
    Retrieve documents

    Args:
        state (dict): The current graph state

    Returns:
        state (dict): New key added to state, documents, that contains retrieved documents
    """
    print("---RETRIEVE---")
    question = state["question"]

    documents = retriever.invoke(question)

    return {"documents":documents, "question":question}

def generate(state):
    """ generate answer 
    
    args: 
    state(dict) the currrent state graph 
    
    return:
    new key added to state, generation, that contains llm generations
    """
    print("---GENERATE---")

    question = state["question"]
    documents = state["documents"]

    rag_chain = prompt | llm | StrOutputParser()
    generation = rag_chain.invoke({"context":documents, "question":question})
    return {"documents": documents, "question": question, "generation": generation}

def grade_documents(state):
    """ Determine whether the retrieved documents are relevant to the question 
    
    args:
    current graph state
    
    returns:
    update documents with filtered documents """

    print("---CHECK DOCUMENT RELEVANCE TO QUESTION---")
    question = state["question"]
    documents = state["documents"]

    filtered_docs = []
    for d in documents:
        score = retrieval_grade.invoke({"question": question, "document": d.page_content})

        grade = score.binary_score

        if grade == "yes":
            print("---GRADE: DOCUMENT RELEVANT---")
            filtered_docs.append(d)
        else:
             print("---GRADE: DOCUMENT NOT RELEVANT---")
             continue
    return {"documents": filtered_docs, "question": question}

def transform_query(state):
    """ Transform the query to produce a better question 
    args:
    the current state graph
    
    returns:
    update question key with rephrased question """

    print("---TRANSFORM QUERY---")
    question = state["question"]
    documents = state["documents"]
    loop_count = state.get("loop_count", 0) + 1

    better_question = question_rewriter_chain.invoke({"question": question})
    return {"documents": documents, "question": better_question, "loop_count": loop_count}

def web_search(state):
    """ web search based on re-phased question 
    args:
    the current graph state
    
    returns:
    Updates documents key with appended web results"""

    print("---WEB SEARCH---")

    question = state["question"]

    docs = web_search_tool.invoke({"query": question})
    web_results = "\n".join([d["content"] for d in docs])
    web_results = Document(page_content=web_results)

    return {"documents": [web_results], "question": question}

def route_question(state):
    """ Route question to web search or RAG 
    
    args:
    the current state graph
    
    returns:
    the next node to call """

    print("---ROUTE QUESTION---")
    question = state["question"]
    source = question_router.invoke({"question": question})
    if source.datasource == "websearch":
        print("---ROUTE QUESTION TO WEB SEARCH---")
        return "web_search"
    else:
        print("---ROUTE QUESTION TO RAG---")
        return "vectorstore"

def decide_to_generate(state):
    """ Determine whether to generate an answer or transform the query 
    args:
    the current graph state
    
    response:
    binary decision for next node to call """

    print("---ASSESS GRADED DOCUMENTS---")
    filtered_documents = state["documents"]
    loop_count = state.get("loop_count", 0)

    if not filtered_documents:
        if loop_count >= MAX_RETRIES:
            print("---DECISION: MAX RETRIES REACHED, GENERATE WITH WHAT WE HAVE---")
            return "generate"
        # All documents have been filtered check_relevance
        # We will re-generate a new query
        print(
            "---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---"
        )
        return "transform_query"
    else:
        # We have relevant documents, so generate answer
        print("---DECISION: GENERATE---")
        return "generate"

def grade_generation_v_documents_and_question(state):
    """
    Determines whether the generation is grounded in the document and answers question.

    Args:
        state (dict): The current graph state

    Returns:
        str: Decision for next node to call
    """

    print("---CHECK HALLUCINATIONS---")
    question = state["question"]
    documents = state["documents"]
    generation = state["generation"]
    loop_count = state.get("loop_count", 0)

    score = hallucination_grader.invoke(
        {"document": documents, "generation": generation}
    )
    grade = score.binary_score

    # Check hallucination
    if grade == "yes":
        print("---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---")
        # Check question-answering
        print("---GRADE GENERATION vs QUESTION---")
        score = answer_grader.invoke({"question": question, "generation": generation})
        grade = score.binary_score
        if grade == "yes":
            print("---DECISION: GENERATION ADDRESSES QUESTION---")
            return "useful"
        elif loop_count >= MAX_RETRIES:
            print("---DECISION: MAX RETRIES REACHED, ACCEPTING GENERATION---")
            return "useful"
        else:
            print("---DECISION: GENERATION DOES NOT ADDRESS QUESTION---")
            return "not useful"
    else:
        if loop_count >= MAX_RETRIES:
            pprint("---DECISION: MAX RETRIES REACHED, ACCEPTING GENERATION---")
            return "useful"
        pprint("---DECISION: GENERATION IS NOT GROUNDED IN DOCUMENTS, RE-TRY---")
        return "not supported"
   
    

In [35]:
from langgraph.graph import StateGraph, START, END

workflow = StateGraph(GraphState)

workflow.add_node("web_search", web_search)  # web search
workflow.add_node("retrieve", retrieve)  # retrieve
workflow.add_node("grade_documents", grade_documents)  # grade documents
workflow.add_node("generate", generate)  # generate
workflow.add_node("transform_query", transform_query)

workflow.add_conditional_edges(START, route_question, {"web_search": "web_search", "vectorstore":"retrieve"})
workflow.add_edge("web_search", "generate")
workflow.add_edge("retrieve", "grade_documents")
workflow.add_conditional_edges("grade_documents", decide_to_generate, {"transform_query": "transform_query", "generate":"generate"})
workflow.add_edge("transform_query", "retrieve")
workflow.add_conditional_edges("generate", grade_generation_v_documents_and_question, {"not supported": "generate",
        "useful": END,
        "not useful": "transform_query",})

app = workflow.compile()


In [36]:
app.invoke({"question":"What is machine learning"})

---ROUTE QUESTION---
---ROUTE QUESTION TO WEB SEARCH---
---WEB SEARCH---
---GENERATE---
---CHECK HALLUCINATIONS---
---DECISION: GENERATION IS GROUNDED IN DOCUMENTS---
---GRADE GENERATION vs QUESTION---
---DECISION: GENERATION DOES NOT ADDRESS QUESTION---
---TRANSFORM QUERY---
---RETRIEVE---
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---
---TRANSFORM QUERY---
---RETRIEVE---
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---GRADE: DOCUMENT NOT RELEVANT---
---ASSESS GRADED DOCUMENTS---
---DECISION: ALL DOCUMENTS ARE NOT RELEVANT TO QUESTION, TRANSFORM QUERY---
---TRANSFORM QUERY---
---RETRIEVE---
---CHECK DOCUMENT RELEVANCE TO QUESTION---
---GRADE: DOC

{'question': '**Improved question:**  \nWhat is machine learning—how is it defined, what are its core principles, which main techniques does it encompass, and what are common real‑world applications of these techniques?',
 'generation': 'Machine learning is a branch of artificial intelligence that builds statistical models from data so computers can make predictions or decisions without explicit programming. Its core principles are learning from examples, generalizing to new cases, and optimizing model performance, and it includes techniques such as supervised, unsupervised, reinforcement, and deep learning. Real‑world applications span image and speech recognition, recommendation systems, fraud detection, autonomous vehicles, and natural‑language processing.',
 'documents': [],
 'loop_count': 3}